In [1]:
import torch
import torchvision

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

PyTorch: 2.13.0+cpu
Torchvision: 0.28.0+cpu


# Practice 2 – Mô hình ResNet-18

Notebook này thực hiện hai chiến lược huấn luyện:

- ResNet-18 A: đóng băng backbone và chỉ huấn luyện lớp `fc`.
- ResNet-18 B: fine-tuning `layer4` và lớp `fc`.

Hai thí nghiệm sẽ sử dụng cùng learning rate, batch size, optimizer và số epoch để bảo đảm so sánh công bằng.

## 1. Mục tiêu

- Tải mô hình ResNet-18 pre-trained.
- Khảo sát kiến trúc mô hình.
- Xác định lớp phân loại cuối `model.fc`.
- Thay lớp cuối để đầu ra có 10 lớp.
- Thực hiện transfer learning bằng cách chỉ huấn luyện lớp `fc`.
- Thực hiện fine-tuning bằng cách mở `layer4` và `fc`.
- Đếm tổng số tham số và số tham số được huấn luyện.
- Kiểm tra kích thước đầu ra của mô hình.

In [11]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()

if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Thư mục notebook:", NOTEBOOK_DIR)
print("Thư mục Practice_2:", PROJECT_ROOT)
print("Thư mục src tồn tại:", (PROJECT_ROOT / "src").exists())
print(
    "File resnet18.py tồn tại:",
    (PROJECT_ROOT / "src" / "models" / "resnet18.py").exists()
)

Thư mục notebook: D:\DEEP\UTH-Deep-Learning-nhom2\Practice_2\notebooks
Thư mục Practice_2: D:\DEEP\UTH-Deep-Learning-nhom2\Practice_2
Thư mục src tồn tại: True
File resnet18.py tồn tại: True


In [12]:
import torch
import pandas as pd

from src.models.resnet18 import (
    build_resnet18_fc_only,
    build_resnet18_finetune_layer4,
    count_parameters,
    get_trainable_parameter_names
)

print("Import thư viện và mô hình thành công.")

Import thư viện và mô hình thành công.


## 2. Thiết lập thiết bị

Mô hình sẽ sử dụng GPU nếu CUDA khả dụng. Nếu không có GPU, chương trình sẽ tự động sử dụng CPU.

In [13]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Thiết bị đang sử dụng:", device)

Thiết bị đang sử dụng: cpu


## 3. ResNet-18 A – Chỉ huấn luyện lớp fc

Trong thí nghiệm này, toàn bộ backbone của ResNet-18 được đóng băng. Chỉ lớp phân loại cuối `model.fc` được cập nhật trọng số.

Đây là chiến lược transfer learning, trong đó mô hình sử dụng các đặc trưng đã học từ ImageNet và chỉ huấn luyện classifier mới cho CIFAR-10.

In [14]:
model_a = build_resnet18_fc_only(
    num_classes=10,
    use_pretrained=True
)

model_a = model_a.to(device)

print("Đã tạo ResNet-18 A.")

Đã tạo ResNet-18 A.


In [15]:
print(model_a)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

### Phân tích kiến trúc ResNet-18

- `conv1`: lớp tích chập đầu tiên, nhận ảnh đầu vào.
- `bn1`: chuẩn hóa đầu ra của lớp tích chập.
- `relu`: hàm kích hoạt phi tuyến.
- `maxpool`: giảm kích thước không gian của đặc trưng.
- `layer1` đến `layer4`: các nhóm residual block dùng để trích xuất đặc trưng.
- `avgpool`: gom các đặc trưng trước khi phân loại.
- `fc`: lớp phân loại cuối của ResNet-18.

In [16]:
print("Lớp phân loại cuối:")
print(model_a.fc)

print("\nSố đặc trưng đầu vào:", model_a.fc.in_features)
print("Số lớp đầu ra:", model_a.fc.out_features)

Lớp phân loại cuối:
Linear(in_features=512, out_features=10, bias=True)

Số đặc trưng đầu vào: 512
Số lớp đầu ra: 10


Lớp phân loại cuối của ResNet-18 là `model.fc`. Lớp gốc đã được thay bằng `nn.Linear(512, 10)` để phù hợp với 10 lớp của bộ dữ liệu CIFAR-10.

In [17]:
params_a = count_parameters(model_a)

print(f"Tổng số tham số: {params_a['total']:,}")
print(f"Số tham số trainable: {params_a['trainable']:,}")
print(f"Số tham số đóng băng: {params_a['frozen']:,}")

Tổng số tham số: 11,181,642
Số tham số trainable: 5,130
Số tham số đóng băng: 11,176,512


In [18]:
trainable_names_a = get_trainable_parameter_names(model_a)

print("Các tham số được huấn luyện:")

for name in trainable_names_a:
    print("-", name)

Các tham số được huấn luyện:
- fc.weight
- fc.bias


In [19]:
assert trainable_names_a == [
    "fc.weight",
    "fc.bias"
]

print("ResNet-18 A đã đóng băng đúng.")

ResNet-18 A đã đóng băng đúng.


Kiểm tra đầu ra mô hình A

Chưa có dữ liệu của Tấn Lên nên ta dùng dữ liệu giả.

In [20]:
dummy_images = torch.randn(
    2,
    3,
    224,
    224,
    device=device
)

model_a.eval()

with torch.no_grad():
    outputs_a = model_a(dummy_images)

print("Kích thước đầu vào:", dummy_images.shape)
print("Kích thước đầu ra:", outputs_a.shape)

Kích thước đầu vào: torch.Size([2, 3, 224, 224])
Kích thước đầu ra: torch.Size([2, 10])


In [21]:
assert outputs_a.shape == (2, 10)

print("Đầu ra ResNet-18 A phù hợp với CIFAR-10.")

Đầu ra ResNet-18 A phù hợp với CIFAR-10.


## 4. ResNet-18 B – Fine-tuning layer4 và fc

Trong thí nghiệm này, các lớp từ đầu mô hình đến `layer3` vẫn được đóng băng. Riêng `layer4` và lớp `fc` được mở để cập nhật trọng số.

Việc mở `layer4` giúp mô hình điều chỉnh các đặc trưng cấp cao theo dữ liệu CIFAR-10.

In [22]:
model_b = build_resnet18_finetune_layer4(
    num_classes=10,
    use_pretrained=True
)

model_b = model_b.to(device)

print("Đã tạo ResNet-18 B.")

Đã tạo ResNet-18 B.


In [23]:
print("Lớp phân loại cuối:")
print(model_b.fc)

print("\nSố lớp đầu ra:", model_b.fc.out_features)

Lớp phân loại cuối:
Linear(in_features=512, out_features=10, bias=True)

Số lớp đầu ra: 10


In [24]:
params_b = count_parameters(model_b)

print(f"Tổng số tham số: {params_b['total']:,}")
print(f"Số tham số trainable: {params_b['trainable']:,}")
print(f"Số tham số đóng băng: {params_b['frozen']:,}")

Tổng số tham số: 11,181,642
Số tham số trainable: 8,398,858
Số tham số đóng băng: 2,782,784


In [25]:
trainable_names_b = get_trainable_parameter_names(model_b)

print("Số tensor tham số trainable:", len(trainable_names_b))

print("\n10 tham số đầu tiên:")
for name in trainable_names_b[:10]:
    print("-", name)

print("\n5 tham số cuối:")
for name in trainable_names_b[-5:]:
    print("-", name)

Số tensor tham số trainable: 17

10 tham số đầu tiên:
- layer4.0.conv1.weight
- layer4.0.bn1.weight
- layer4.0.bn1.bias
- layer4.0.conv2.weight
- layer4.0.bn2.weight
- layer4.0.bn2.bias
- layer4.0.downsample.0.weight
- layer4.0.downsample.1.weight
- layer4.0.downsample.1.bias
- layer4.1.conv1.weight

5 tham số cuối:
- layer4.1.conv2.weight
- layer4.1.bn2.weight
- layer4.1.bn2.bias
- fc.weight
- fc.bias


In [26]:
invalid_names_b = [
    name
    for name in trainable_names_b
    if not (
        name.startswith("layer4.")
        or name.startswith("fc.")
    )
]

assert len(invalid_names_b) == 0
assert "fc.weight" in trainable_names_b
assert "fc.bias" in trainable_names_b

print("ResNet-18 B đã mở đúng layer4 và fc.")

ResNet-18 B đã mở đúng layer4 và fc.


In [27]:
model_b.eval()

with torch.no_grad():
    outputs_b = model_b(dummy_images)

print("Kích thước đầu ra:", outputs_b.shape)

assert outputs_b.shape == (2, 10)

print("Đầu ra ResNet-18 B phù hợp với CIFAR-10.")

Kích thước đầu ra: torch.Size([2, 10])
Đầu ra ResNet-18 B phù hợp với CIFAR-10.


## 5. So sánh số tham số của hai chiến lược

In [28]:
parameter_comparison = pd.DataFrame({
    "Thí nghiệm": [
        "ResNet-18 A",
        "ResNet-18 B"
    ],
    "Chiến lược": [
        "Chỉ huấn luyện fc",
        "Fine-tuning layer4 và fc"
    ],
    "Tổng tham số": [
        params_a["total"],
        params_b["total"]
    ],
    "Tham số trainable": [
        params_a["trainable"],
        params_b["trainable"]
    ],
    "Tham số đóng băng": [
        params_a["frozen"],
        params_b["frozen"]
    ]
})

parameter_comparison

,Thí nghiệm,Chiến lược,Tổng tham số,Tham số trainable,Tham số đóng băng
0,ResNet-18 A,Chỉ huấn luyện fc,11181642,5130,11176512
1,ResNet-18 B,Fine-tuning layer4 và fc,11181642,8398858,2782784


## 6. Nhận xét ban đầu

ResNet-18 A chỉ cập nhật trọng số của lớp `fc`, do đó số tham số trainable rất nhỏ so với tổng số tham số của mô hình. Chiến lược này giúp giảm thời gian và chi phí huấn luyện.

ResNet-18 B mở thêm `layer4`, nên số tham số trainable lớn hơn đáng kể. Mô hình có khả năng điều chỉnh các đặc trưng cấp cao theo dữ liệu CIFAR-10, nhưng thời gian huấn luyện và mức sử dụng tài nguyên dự kiến cũng sẽ cao hơn.

Hiệu quả thực tế của hai chiến lược sẽ được kết luận sau khi huấn luyện trong cùng một cấu hình và so sánh validation loss, validation accuracy và thời gian huấn luyện.

## 7. Cấu hình huấn luyện

## 8. Huấn luyện ResNet-18 A

## 9. Huấn luyện ResNet-18 B

## 10. So sánh kết quả huấn luyện

## 11. Lưu checkpoint tốt nhất

## 12. Kết luận transfer learning và fine-tuning